# 03 - Structured invoice extraction

Pydantic schemas make agent output machine-readable. The output schema constrains the shape; application validation still enforces business rules.

**Runtime:** ADK 2.10.0, your existing `dataengenv` kernel and Application Default Credentials (ADC).

**Cost:** Two small Gemini calls. All invoice data is synthetic. Run top to bottom; no `course.py` or `config.json` is needed.

## Initialize

Set your project, location and model below. As in notebooks 01 and 02, the client uses standard TLS certificate verification.

In [ ]:
from google.adk.agents import Agent
from google.adk.models import Gemini
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

PROJECT = "gen-lang-client-0367558705"
LOCATION = "global"
MODEL = "gemini-2.5-flash-lite"
APP_NAME = "ecommerce_lab"
USER_ID = "customer_101"

model = Gemini(
    model=MODEL,
    client_kwargs={"vertexai": True, "project": PROJECT, "location": LOCATION},
)

## Define the output schema

Each invoice must have these fields. `Field(ge=0)` rejects negative amounts during local validation. We check the arithmetic separately after extraction.

In [ ]:
from pydantic import BaseModel, Field, ValidationError

class InvoiceSummary(BaseModel):
    invoice_id: str
    order_id: str
    currency: str
    subtotal: float = Field(ge=0)
    tax: float = Field(ge=0)
    total: float = Field(ge=0)

## Extract the invoice

`output_schema` requests the structured response. `output_key="invoice"` stores the output in session state. The Runner produces events; read the final response and validate it with Pydantic.

In [ ]:
agent = Agent(
    name="invoice_parser",
    model=model,
    instruction="Extract exactly the supplied invoice facts. Do not calculate or guess missing identifiers.",
    output_schema=InvoiceSummary,
    output_key="invoice",
    generate_content_config=types.GenerateContentConfig(
        temperature=0, max_output_tokens=768,
    ),
)

session_service = InMemorySessionService()
session = await session_service.create_session(app_name=APP_NAME, user_id=USER_ID)
runner = Runner(agent=agent, app_name=APP_NAME, session_service=session_service)

message = types.Content(
    role="user",
    parts=[types.Part(
        text="Invoice INV1001 for order O1001: subtotal INR 3497, GST 629.46, total 4126.46."
    )],
)
events = []
invoice_json = ""
async for event in runner.run_async(
    user_id=USER_ID, session_id=session.id, new_message=message,
):
    events.append(event)
    if event.error_code:
        raise RuntimeError(f"{event.error_code}: {event.error_message}")
    if event.is_final_response() and event.content:
        invoice_json = "".join(part.text or "" for part in event.content.parts or [])

In [ ]:
invoice = InvoiceSummary.model_validate_json(invoice_json)
if abs(invoice.subtotal + invoice.tax - invoice.total) >= 0.01:
    raise ValueError("Invoice total does not match subtotal plus tax")
print(invoice.model_dump())

# Fetch the updated session after the Runner has stored the output.
session = await session_service.get_session(app_name=APP_NAME, user_id=USER_ID, session_id=session.id)
print("Stored output:", session.state["invoice"])

## Validate input before sending it

An `input_schema` describes the agent's expected input. Build and validate the request locally, then send its JSON representation. This example uses a separate session for the second agent.

In [ ]:
class OrderQuestion(BaseModel):
    order_id: str = Field(pattern=r"^O\d{4}$")
    question: str

request = OrderQuestion(order_id="O1001", question="What is the total?")
print(request.model_dump_json())

In [ ]:
typed_agent = Agent(
    name="typed_request",
    model=model,
    instruction="Read the JSON order question and restate it briefly.",
    input_schema=OrderQuestion,
    generate_content_config=types.GenerateContentConfig(
        temperature=0, max_output_tokens=768,
    ),
)
typed_session = await session_service.create_session(app_name=APP_NAME, user_id=USER_ID)
typed_runner = Runner(agent=typed_agent, app_name=APP_NAME, session_service=session_service)

message = types.Content(role="user", parts=[types.Part(text=request.model_dump_json())])
async for event in typed_runner.run_async(
    user_id=USER_ID, session_id=typed_session.id, new_message=message,
):
    if event.error_code:
        raise RuntimeError(f"{event.error_code}: {event.error_message}")
    if event.is_final_response() and event.content:
        answer = "".join(part.text or "" for part in event.content.parts or [])
        print("Final answer:", answer)

## See local validation fail

No model call is needed to reject a negative total or an invalid order ID. Catch the expected errors so the notebook can keep running.

In [ ]:
try:
    InvoiceSummary(
        invoice_id="INV1001", order_id="O1001", currency="INR",
        subtotal=3497, tax=629.46, total=-1,
    )
except ValidationError as error:
    print("Invalid invoice:", error.errors()[0]["msg"])

try:
    OrderQuestion(order_id="UNKNOWN", question="What is the total?")
except ValidationError as error:
    print("Invalid request:", error.errors()[0]["msg"])

## Try it

Add an invoice line-item schema. Change the invoice text and rerun extraction; compare schema validation with the separate arithmetic check.

## Reference

[Official ADK documentation](https://adk.dev/). Configuration is in the first code cell. Use your existing authenticated kernel; no activation or login cells are needed.